# ISOM 839 · Session 3 — Sensitivity, Duality & What-If Thinking

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week the model told you **what to do**: build 30 desks and 40 chairs, earn $4,100, and both departments run at full capacity. Then we asked the question you couldn't answer: *if you could add one more hour of labor, which department gets it?*

Tonight the model tells you **what everything is worth.**

In [ ]:
%pip install -q gurobipy
import gurobipy as gp
from gurobipy import GRB
print('gurobipy ready:', gp.gurobi.version())

---
## Part 1 — The reveal: shadow prices

Same model as last week. One new trick: after solving, every constraint carries an attribute **`.Pi`** — its *shadow price*: how much the optimal profit rises if that constraint's right-hand side increases by one unit.

In [ ]:
m = gp.Model('production_mix')
D = m.addVar(name='desks')
C = m.addVar(name='chairs')
m.setObjective(70*D + 50*C, GRB.MAXIMIZE)
carpentry = m.addConstr(4*D + 3*C <= 240, 'carpentry')
finishing = m.addConstr(2*D + 1*C <= 100, 'finishing')
m.optimize()

print(f'\nPlan: ({D.X:.0f}, {C.X:.0f})  profit ${m.ObjVal:,.0f}')
print(f'One more CARPENTRY hour is worth:  ${carpentry.Pi:.2f}')
print(f'One more FINISHING hour is worth:  ${finishing.Pi:.2f}')

**The cliffhanger is answered: carpentry, and it isn't close — $15/hr vs $5/hr.**

Don't take `.Pi`'s word for it. *Prove it* — give the shop one extra carpentry hour and re-solve:

In [ ]:
carpentry.RHS = 241
m.optimize()
print(f'With 241 carpentry hours: ({D.X:.1f}, {C.X:.1f})  profit ${m.ObjVal:,.2f}   <- was $4,100.00')
carpentry.RHS = 240   # put it back
m.optimize()

$4,115 — exactly $15 more. Notice the *plan* changed too: (29.5, 41). The extra hour is not used to "make more of everything"; the whole mix rebalances.

### 💬 Discussion: the overtime offer
A staffing agency offers overtime at **$25/hour, either department.** How many hours do you buy?

Then the price drops to **$10/hour.** Now what?

*(Write your answer before running anything. Hint: compare the price to the shadow price — and remember what "worth" means at the margin.)*

---
## Part 2 — The fine print: ranging

A shadow price is a **marginal** value — valid only within a range. gurobipy reports the range on every constraint (`SARHSLow`, `SARHSUp`) and on every objective coefficient (`SAObjLow`, `SAObjUp`):

In [ ]:
print('RHS ranging (shadow price valid inside this range):')
for c in [carpentry, finishing]:
    print(f'  {c.ConstrName:10s} Pi ${c.Pi:5.2f}   valid RHS range [{c.SARHSLow:.0f}, {c.SARHSUp:.0f}]')
print('\nObjective ranging (plan stays (30, 40) inside this range):')
for v in [D, C]:
    print(f'  {v.VarName:8s} profit coeff {v.Obj:.0f}   safe range [{v.SAObjLow:.2f}, {v.SAObjUp:.2f}]')

Read it like a manager:
- Carpentry's $15/hr holds from **200 to 300 hours**. Past 300, carpentry stops being the bottleneck (the shop goes all-chairs) and its value collapses to $0.
- Desk profit can swing between **$66.67 and $100** without changing the plan at all. Decisions are *stable within ranges, then jump* — Session 1's corner-hopping, now with exact boundaries.

---
## Part 3 — Which products deserve to exist: reduced costs

Marketing proposes a **bookcase**: $50 profit, 3 hrs carpentry, 2 hrs finishing. Before solving — price its ingredients using shadow prices:

$$3 \text{ hrs} \times \$15 + 2 \text{ hrs} \times \$5 = \$55 \text{ of scarce capacity, for } \$50 \text{ of profit.}$$

In [ ]:
m2 = gp.Model('with_bookcase')
D2 = m2.addVar(name='desks'); C2 = m2.addVar(name='chairs'); B2 = m2.addVar(name='bookcases')
m2.setObjective(70*D2 + 50*C2 + 50*B2, GRB.MAXIMIZE)
m2.addConstr(4*D2 + 3*C2 + 3*B2 <= 240, 'carpentry')
m2.addConstr(2*D2 + 1*C2 + 2*B2 <= 100, 'finishing')
m2.optimize()
print(f'\nPlan: desks {D2.X:.0f}, chairs {C2.X:.0f}, bookcases {B2.X:.0f}')
print(f'Bookcase reduced cost: ${B2.RC:.2f}  (negative = does not deserve capacity yet)')

The solver builds **zero bookcases**, and `.RC` says why: −$5. Every bookcase would *destroy* $5 by stealing capacity from better products.

**Try it:** change the bookcase profit to 56 and re-run. It storms into the plan — and reshapes everything.

### The books always balance (duality in one line)
Value all resources at their shadow prices: 240 × $15 + 100 × $5 = **$4,100** — exactly the optimal profit. Every dollar of profit is accounted for by a scarce resource. That identity (strong duality) is the deepest fact in this course:

In [ ]:
resource_value = carpentry.RHS * carpentry.Pi + finishing.RHS * finishing.Pi
print(f'Total resource value: ${resource_value:,.2f}')
print(f'Optimal profit:       ${m.ObjVal:,.2f}')
print('The books balance:', abs(resource_value - m.ObjVal) < 1e-6)

---
## Part 4 — The lie detector 🕵️

An AI assistant "helpfully" rebuilt your production model. Its output looks plausible: a plan, a profit, everything runs. **But the model below contains one silent error.**

You know two facts from walking the shop floor: *both* departments are slammed — nobody is idle, ever.

Run it, read the sensitivity report, and find the lie.

In [ ]:
ai = gp.Model('ai_generated')
Da = ai.addVar(name='desks'); Ca = ai.addVar(name='chairs')
ai.setObjective(70*Da + 50*Ca, GRB.MAXIMIZE)
ai.addConstr(4*Da + 3*Ca <= 240, 'carpentry')
ai.addConstr(1*Da + 2*Ca <= 100, 'finishing')   # <- the AI wrote this line
ai.optimize()
print(f'\nAI plan: ({Da.X:.0f}, {Ca.X:.0f})  profit ${ai.ObjVal:,.0f}')
for c in ai.getConstrs():
    print(f'  {c.ConstrName:10s} slack {c.Slack:6.1f}   Pi ${c.Pi:.2f}')

**The tell:** the report claims finishing has **40 idle hours** and is worth **$0/hr** — but you've seen the finishing room. It's slammed. A model whose sensitivity story contradicts the physical floor is lying somewhere.

The lie: desks need **2** finishing hours, not 1 — the AI swapped the coefficients. Fix the constraint, re-run, and watch the report line up with reality again.

> This is why AI makes sensitivity analysis MORE valuable, not less: anyone can generate a model now. The analyst who can *audit* one — by checking its economics against the world — is the one who gets trusted.

---
# Homework #2 — The COO Memo 📄

**Due before Session 4 (Wed Sep 30), on Canvas: this notebook completed + a one-page memo.**

You are the analytics lead at **Meridian Manufacturing**. Meridian makes three enclosure products with the margins and resource needs below. Deluxe was **discontinued last quarter** by gut feel; the COO wants a data-backed second opinion on everything.

| | Standard | Deluxe | Premium | Available |
|---|---|---|---|---|
| Margin / unit | $32 | $42 | $60 | |
| Machining hrs | 2 | 3 | 4 | 600 |
| Assembly hrs | 1 | 2 | 3 | 400 |
| Packaging hrs | 1 | 1 | 1 | 180 |

**Build and solve the LP, extract the full sensitivity report, then write a one-page memo to the COO answering:**
1. **Which resource should we expand first, and what is the maximum price per hour worth paying?** (Cite Pi and say how many hours the price holds for — use ranging.)
2. **Does Deluxe deserve a second look?** At what margin would it earn its way back into the plan? (Cite RC.)
3. **How robust is this plan if Premium's margin falls 15%?** (Cite objective ranging — and say precisely where the cliff is.)

Every claim must cite a specific number. The memo itself: plain executive English, zero jargon.

In [ ]:
hw = gp.Model('meridian')

# TODO 1: decision variables (units of Standard, Deluxe, Premium)

# TODO 2: objective — maximize total margin

# TODO 3: three resource constraints

hw.optimize()

# TODO 4: print the plan, then for each constraint: Slack, Pi, SARHSLow, SARHSUp
#         and for each variable: RC, SAObjLow, SAObjUp

---
### Submission checklist
- [ ] All TODO cells complete and running
- [ ] One-page COO memo (PDF or in-notebook markdown), every claim citing a Pi, RC, or range
- [ ] The lie-detector model fixed and re-run

**Next week — Session 4:** the models go to the map. Transportation, transshipment, and the network that feeds 810 million people. 🚚